# batchnorm-affine-params — faded example 2: Recover BatchNorm gamma from two points per channel

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-affine-params`. Running the beacon reports progress on the `CNN: BatchNorm affine params` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm affine params` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-affine-params`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-affine-params"
DD_SUBTOPIC = "CNN: BatchNorm affine params"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

BatchNorm's per-channel affine `y = gamma * x_hat + beta` is a 1-D line. Two distinct `(x_hat, y)` points in a channel determine it: the slope `gamma = (y1 - y0) / (x1 - x0)` and intercept `beta = y0 - gamma * x0`. Channels are independent, so all `C` slopes compute in parallel.

## Faded exercise 2

### Faded — recover gamma (slope) per channel

Implement `recover_gamma(x_hat, y)`. Given `x_hat: (B, C, H, W)` (pre-affine) and `y: (B, C, H, W)` (post-affine), recover `gamma` as a length-`C` tensor by treating each channel as a line through two distinct points.

The einops flatten and the beta computation are given. You must complete the **slope** computation.

**Fill in:** Compute the per-channel slope gamma = (y1 - y0) / (x1 - x0) using the first two flattened entries of each channel.

In [ ]:
def recover_gamma(x_hat: Tensor, y: Tensor):
    x_flat = rearrange(x_hat, 'b c h w -> c (b h w)')
    y_flat = rearrange(y, 'b c h w -> c (b h w)')
    x0, x1 = x_flat[:, 0], x_flat[:, 1]
    y0, y1 = y_flat[:, 0], y_flat[:, 1]
    gamma = (y1 - y0) / (x1 - x0)
    beta = y0 - gamma * x0
    return gamma, beta


def _test():
    t.manual_seed(0)
    B, C, H, W = 3, 4, 2, 2
    x_hat = t.randn(B, C, H, W)
    true_gamma = t.tensor([2.0, 0.5, 1.5, 3.0])
    true_beta = t.tensor([1.0, -1.0, 0.0, 2.0])
    y = true_gamma.view(1, -1, 1, 1) * x_hat + true_beta.view(1, -1, 1, 1)
    gamma, beta = recover_gamma(x_hat, y)
    assert gamma.shape == (C,), gamma.shape
    assert beta.shape == (C,), beta.shape
    assert t.allclose(gamma, true_gamma, atol=1e-4), (gamma, true_gamma)
    assert t.allclose(beta, true_beta, atol=1e-4), (beta, true_beta)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def recover_gamma(x_hat: Tensor, y: Tensor):
    x_flat = rearrange(x_hat, 'b c h w -> c (b h w)')
    y_flat = rearrange(y, 'b c h w -> c (b h w)')
    x0, x1 = x_flat[:, 0], x_flat[:, 1]
    y0, y1 = y_flat[:, 0], y_flat[:, 1]
    gamma = (y1 - y0) / (x1 - x0)
    beta = y0 - gamma * x0
    return gamma, beta
```
</details>